# T539 Closed-Loop AOS State — infocus_initial_alignment (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-07-08  
**Last Modified:** 2026-07-08  
**Status:** In Progress  
**Keywords:** Rubin Observatory, AOS, closed loop, MTAOS, wavefront, degree of freedom, trim, T539

## Description

Locate the in-focus images taken at the end of BLOCK-T539 closed-loop alignment
sequences and collect their delivered image quality and AOS state.

Selection criteria:
- `science_program` LIKE `BLOCK-T539%`
- `observation_reason` == `infocus_initial_alignment`
- observation annotation (`scheduler_note`) contains `closed_loop_22dof_trunc12`
- `day_obs >= day_obs_min`

For each matched image we collect:
1. **PSF FWHM median** — ConsDB `visit1_quicklook.psf_sigma_median`
2. **Donut blur FWHM** and **residual AOS FWHM** — ConsDB quicklook (if present)
3. **Aggregated Degree-of-Freedom vector** (total applied Trim / offset from LUT) —
   reuses `aos/code/aos_trim.py`, which anchors on the ConsDB exposure `obs_start`
   and pulls the `MTAOS.logevent_degreeOfFreedom` event in effect before each exposure
4. **Retrieved wavefront (OPD)** per corner WFS — EFD `MTAOS.logevent_wavefrontError`,
   Zernikes Z4..Z26 excluding Z20, Z21, anchored on the same `obs_start`

**Output:** `output/t539_closedloop_aos_{day_obs_min}_{day_obs_max}.parquet`

**Based on:** locate_test_blocks.ipynb (ConsDB), aos/code/aos_trim.py (DOF/Trim), nightly_tablemaker (obs_start anchoring)

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-08 | Aaron Roodman | Initial version |
| 2026-07-08 | Aaron Roodman | Reuse aos_trim for DOF; fix ConsDB query to use v1.* |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs_min = 20260420
day_obs_max = 20260708          # bump to today when re-running

instrument = 'lsstcam'
science_program_like = 'BLOCK-T539%'
observation_reason = 'infocus_initial_alignment'
annotation_contains = 'closed_loop_22dof_trunc12'

consdb_url = 'http://consdb-pq.consdb:8080/consdb'

# Corner wavefront sensors (SW0 half-chips): detector id -> raft name
corners = {191: 'R00_SW0', 195: 'R04_SW0', 199: 'R40_SW0', 203: 'R44_SW0'}

# Zernikes to keep: Z4..Z26 excluding Z20, Z21 (Noll) — 21 coefficients
zk_noll = [z for z in range(4, 27) if z not in (20, 21)]

# Wavefront-error lookback window before obs_start (seconds): the closed-loop
# iteration that produced this image publishes wavefrontError just before it.
wf_lookback = 180.0

output_dir = 'output'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

os.environ["no_proxy"] += ",.consdb"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.time import Time, TimeDelta
from astropy.table import Table

from tqdm.notebook import tqdm

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
setup_plotting()

# Reuse the AOS trim / DOF helpers (aos/code/aos_trim.py)
sys.path.insert(0, str(Path.cwd().parent / 'aos' / 'code'))
import aos_trim
from lsst.summit.utils.efdUtils import getMostRecentRowWithDataBefore

<a id='functions'></a>
## Helper Functions

In [ ]:
WF_TOPIC = 'lsst.sal.MTAOS.logevent_wavefrontError'


def fetch_corner_zernikes(efd_client, obs_start_list, lookback=wf_lookback):
    """Per-visit retrieved-wavefront Zernikes for the 4 corner WFS.

    For each exposure, look back `lookback` seconds from `obs_start` (TAI)
    and take the most recent `wavefrontError` row per corner sensor. Uses
    the same obs_start anchoring as aos_trim / nightly_tablemaker.

    Parameters
    ----------
    efd_client : EfdClient
    obs_start_list : list of str or None
        TAI isot exposure-start strings (None where ConsDB had no match).
    lookback : float
        Seconds before obs_start to search for wavefrontError events.

    Returns
    -------
    list of dict
        One dict per visit with keys `z{noll}_{corner}` (nm). Missing
        values are NaN.
    """
    rows = []
    for obs_start in obs_start_list:
        out = {}
        if obs_start is not None:
            t_end = Time(obs_start, format='isot', scale='tai').utc
            t_beg = t_end - TimeDelta(lookback, format='sec')
            try:
                wf = efd_client.select_time_series(
                    WF_TOPIC, ['*'], t_beg, t_end)
            except Exception:
                wf = None
            if wf is not None and len(wf) > 0 and 'sensorId' in wf.columns:
                for det, name in corners.items():
                    sub = wf[wf['sensorId'] == det]
                    if len(sub) == 0:
                        continue
                    r = sub.iloc[-1]
                    for z in zk_noll:
                        # Default: array contiguous from Z4 -> index z-4.
                        # VERIFY with the inspect cell below.
                        col = f"zernikeEstimated{z - 4}"
                        out[f"z{z}_{name}"] = r[col] if col in wf.columns else np.nan
        rows.append(out)
    return rows

<a id='data'></a>
## Data Access

In [ ]:
# Select the matching in-focus closed-loop images from ConsDB.
# Use v1.* (naming individual visit1 columns triggers a 500 when one is absent).
client = aos_trim.make_consdb_client(consdb_url)

visits_query = f'''
    SELECT v1.*,
           ql.physical_rotator_angle,
           ql.psf_sigma_median,
           ql.seeing_zenith_500nm_median
    FROM cdb_{instrument}.visit1 v1
    LEFT JOIN cdb_{instrument}.visit1_quicklook ql
      ON v1.visit_id = ql.visit_id
    WHERE v1.science_program LIKE '{science_program_like}'
      AND v1.observation_reason = '{observation_reason}'
      AND v1.day_obs >= {day_obs_min} AND v1.day_obs <= {day_obs_max}
'''
visits = client.query(visits_query).to_pandas()

# Observation annotation lives in scheduler_note
visits = visits[visits['scheduler_note'].str.contains(
    annotation_contains, na=False)].copy()
visits = visits.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)

# PSF FWHM median (arcsec) from psf_sigma_median (pixels, 0.2"/pix)
visits['psf_fwhm_median'] = 2.355 * visits['psf_sigma_median'] * 0.2

print(f"Matched {len(visits)} images")
visits[['day_obs', 'seq_num', 'band', 'psf_fwhm_median', 'scheduler_note']].head(20)

In [ ]:
# Donut blur + residual AOS FWHM (may or may not be in the quicklook schema)
if len(visits) > 0:
    ids = ",".join(str(v) for v in visits['visit_id'])
    for col in ('donut_blur_fwhm', 'aos_fwhm'):
        try:
            q = f'''SELECT visit_id, {col} FROM cdb_{instrument}.visit1_quicklook
                    WHERE visit_id IN ({ids})'''
            extra = client.query(q).to_pandas()
            visits = visits.merge(extra, on='visit_id', how='left')
            print(f"Added {col}")
        except Exception as e:
            print(f"{col} not available in ConsDB: {type(e).__name__}")

In [ ]:
# ---- Inspect EFD wavefrontError field naming before trusting the mapping ----
# Run once; paste the output back if the field names differ from assumptions.
# (DOF field naming is handled inside aos_trim, which is already validated.)
efd = aos_trim.make_efd_client()

_t0 = Time('2026-04-20T00:00:00')
_t1 = Time('2026-04-30T00:00:00')
wf_probe = efd.select_time_series(WF_TOPIC, ['*'], _t0, _t1)
print(f"wavefrontError rows in probe window: {len(wf_probe)}")
print("\nzernike columns:")
print([c for c in wf_probe.columns if 'ernike' in c])
print("\nother columns:")
print([c for c in wf_probe.columns if 'ernike' not in c])

<a id='analysis'></a>
## Analysis

In [ ]:
# Aggregated DOF (total applied Trim / offset from LUT), reusing aos_trim.
# It anchors on the ConsDB exposure obs_start and takes the degreeOfFreedom
# event in effect just before each exposure. Standard 50-DOF ordering.
fit_table = Table.from_pandas(visits[['day_obs', 'seq_num']].astype(int))

trim, dof_info = aos_trim.fetch_aggregated_dof_for_visits(
    fit_table, efd_client=efd, consdb_client=client)
print(f"DOF anchored on obs_start: {dof_info['n_obs_start']}, "
      f"with finite result: {dof_info['n_dof']} / {len(visits)}")

for i in range(aos_trim.N_DOF):
    visits[f'dof{i}'] = trim[:, i]

In [ ]:
# Retrieved wavefront (OPD) per corner, anchored on the same obs_start.
obs_start_list = aos_trim.fetch_obs_start(
    client, visits['day_obs'].astype(int), visits['seq_num'].astype(int))

zk_rows = fetch_corner_zernikes(efd, obs_start_list, lookback=wf_lookback)
zk_df = pd.DataFrame(zk_rows, index=visits.index)
visits = pd.concat([visits, zk_df], axis=1)

n_zk = len([c for c in visits.columns if c.startswith('z') and '_SW0' in c])
print(f"Added {n_zk} per-corner Zernike columns "
      f"({len(zk_noll)} Zernikes x {len(corners)} corners)")

<a id='results'></a>
## Results & Plots

In [ ]:
# Summary of delivered image quality per visit
show = ['day_obs', 'seq_num', 'band', 'psf_fwhm_median',
        'donut_blur_fwhm', 'aos_fwhm', 'altitude', 'physical_rotator_angle']
show = [c for c in show if c in visits.columns]
visits[show]

In [ ]:
# Write full table (metadata + IQ + DOF + per-corner Zernikes) to parquet
os.makedirs(output_dir, exist_ok=True)
out_file = os.path.join(
    output_dir, f't539_closedloop_aos_{day_obs_min}_{day_obs_max}.parquet')
visits.to_parquet(out_file, index=False)
print(f"Wrote {len(visits)} rows x {visits.shape[1]} cols to {out_file}")